In [3]:
from rating import rating_update
rating_update([3.5], [4.0], games_a=4, games_b=3)


([3.552935251260251], [3.947064748739749])

In [30]:
def print_match_result(
    name: str,
    team_a: list[float],
    team_b: list[float],
    games_a: int,
    games_b: int,
    **kwargs
):
    from rating import rating_update, expected_game_share, observed_game_share, team_rating

    ra_before = team_a[0]
    rb_before = team_rating(team_b)

    ra_team = team_rating(team_a)
    rb_team = team_rating(team_b)

    expected = expected_game_share(ra_team, rb_team, kwargs.get("scale", 0.45))
    observed = observed_game_share(games_a, games_b)

    new_a, new_b = rating_update(
        team_a,
        team_b,
        games_a,
        games_b,
        **kwargs
    )

    ra_after = new_a[0]
    delta = ra_after - ra_before

    print(f"Player: {name}")
    print(f"Games: {games_a}-{games_b}")
    print(f"Team rating before: {ra_team:.3f} vs {rb_team:.3f}")
    print(f"Expected game share: {expected:.3f}")
    print(f"Observed game share: {observed:.3f}")
    print(f"Rating before: {ra_before:.3f}")
    print(f"Rating after:  {ra_after:.3f}")
    print(f"Change:        {delta:+.3f}")
    print("-" * 40)


In [33]:
christi = 3.69
partner = 3.74

opponents = [3.83, 3.81]

print_match_result(
    name="Christi Witt",
    team_a=[christi, partner],
    team_b=opponents,
    games_a=11,   
    games_b=10,
)


Player: Christi Witt
Games: 11-10
Team rating before: 3.715 vs 3.820
Expected game share: 0.442
Observed game share: 0.524
Rating before: 3.690
Rating after:  3.700
Change:        +0.010
----------------------------------------


In [1]:
from court_assignment import generate_courts, courts_to_json
from player import Player

players = [
    Player(1, "A", 4.1, 20),
    Player(2, "B", 4.0, 18),
    Player(3, "C", 3.9, 15),
    Player(4, "D", 3.9, 14),
    Player(5, "E", 3.8, 12),
    Player(6, "F", 3.8, 11),
    Player(7, "G", 3.7, 10),
    Player(8, "H", 3.7, 9),
    Player(9, "I", 3.6, 8),
    Player(10, "J", 3.6, 7),
    Player(11, "K", 3.5, 6),
    Player(12, "L", 3.5, 5),
]

courts = generate_courts(players, mode="blended", num_courts=3)
json_data = courts_to_json(courts)

json_data


{'courts': [{'court_number': 1,
   'teams': [{'team_number': 1,
     'players': [{'player_id': 1, 'first_name': 'A', 'rating': 4.1},
      {'player_id': 6, 'first_name': 'F', 'rating': 3.8}]},
    {'team_number': 2,
     'players': [{'player_id': 2, 'first_name': 'B', 'rating': 4.0},
      {'player_id': 5, 'first_name': 'E', 'rating': 3.8}]}]},
  {'court_number': 2,
   'teams': [{'team_number': 1,
     'players': [{'player_id': 3, 'first_name': 'C', 'rating': 3.9},
      {'player_id': 8, 'first_name': 'H', 'rating': 3.7}]},
    {'team_number': 2,
     'players': [{'player_id': 4, 'first_name': 'D', 'rating': 3.9},
      {'player_id': 7, 'first_name': 'G', 'rating': 3.7}]}]},
  {'court_number': 3,
   'teams': [{'team_number': 1,
     'players': [{'player_id': 9, 'first_name': 'I', 'rating': 3.6},
      {'player_id': 12, 'first_name': 'L', 'rating': 3.5}]},
    {'team_number': 2,
     'players': [{'player_id': 10, 'first_name': 'J', 'rating': 3.6},
      {'player_id': 11, 'first_name': '

In [7]:
from db import init_db

init_db()
print("Database initialized")


Database initialized


In [3]:
import sqlite3

conn = sqlite3.connect("tennis_app.db")
cur = conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
cur.fetchall()


[('leagues',),
 ('sqlite_sequence',),
 ('clubs',),
 ('players',),
 ('court_assignments',),
 ('courts',),
 ('matches',),
 ('match_players',)]

In [10]:
from db import (
    create_match,
    add_player_to_match,
    record_match_score,
    get_players_for_league,
)

match_id = create_match(court_id=1)

add_player_to_match(match_id, player_id=1, team="A")
add_player_to_match(match_id, player_id=2, team="A")
add_player_to_match(match_id, player_id=3, team="B")
add_player_to_match(match_id, player_id=4, team="B")


In [5]:
record_match_score(
    match_id,
    games_team_a=11,
    games_team_b=9,
    display_score="6-4, 5-5"
)


ValueError: Team must have at least one player

In [11]:
import sqlite3

conn = sqlite3.connect("tennis_app.db")
conn.row_factory = sqlite3.Row

rows = conn.execute(
    """
    SELECT match_id, player_id, team
    FROM match_players
    WHERE match_id = ?
    """,
    (match_id,)
).fetchall()

[dict(row) for row in rows]


[{'match_id': 4, 'player_id': 1, 'team': 'A'},
 {'match_id': 4, 'player_id': 2, 'team': 'A'},
 {'match_id': 4, 'player_id': 3, 'team': 'B'},
 {'match_id': 4, 'player_id': 4, 'team': 'B'}]

In [12]:
rows = conn.execute(
    """
    SELECT
        mp.team,
        mp.player_id,
        p.current_rating
    FROM match_players mp
    JOIN players p ON mp.player_id = p.id
    WHERE mp.match_id = ?
    """,
    (match_id,)
).fetchall()

[dict(row) for row in rows]


[]

In [3]:
from db import create_match, add_player_to_match, record_match_score

match_id = create_match(court_id=1)

add_player_to_match(match_id, 1, "A")
add_player_to_match(match_id, 2, "A")
add_player_to_match(match_id, 3, "B")
add_player_to_match(match_id, 4, "B")

record_match_score(match_id, 11, 9)


DEBUG TEAM A: []
DEBUG TEAM B: []


RuntimeError: DEBUG: One or both teams empty

In [1]:
import db


USING DATABASE FILE: /Users/kendallwitt/Downloads/TennisApp/tennis_app.db


In [2]:
from db import create_match, add_player_to_match, record_match_score


In [3]:
match_id = create_match(court_id=1)

add_player_to_match(match_id, 1, "A")
add_player_to_match(match_id, 2, "A")
add_player_to_match(match_id, 3, "B")
add_player_to_match(match_id, 4, "B")


In [4]:
import sqlite3
from db import DB_PATH

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

rows = conn.execute(
    "SELECT match_id, player_id, team FROM match_players"
).fetchall()

[dict(r) for r in rows]


[{'match_id': 1, 'player_id': 1, 'team': 'A'},
 {'match_id': 1, 'player_id': 2, 'team': 'A'},
 {'match_id': 1, 'player_id': 3, 'team': 'B'},
 {'match_id': 1, 'player_id': 4, 'team': 'B'},
 {'match_id': 2, 'player_id': 1, 'team': 'A'},
 {'match_id': 2, 'player_id': 2, 'team': 'A'},
 {'match_id': 2, 'player_id': 3, 'team': 'B'},
 {'match_id': 2, 'player_id': 4, 'team': 'B'}]

In [8]:
conn = sqlite3.connect("tennis_app.db")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())


[('leagues',), ('sqlite_sequence',), ('clubs',), ('players',), ('court_assignments',), ('courts',), ('matches',), ('match_players',), ('court_players',)]
